
# 03G_v2 — Fireline Temporal Production Reconstruction

**Notebook build ID: `03G_TEMPORAL_PRODUCTION_V2`**

This notebook moves from benchmarking to the first full production reconstruction.

The methodological choices established in the earlier audits are:

\[
\boxed{L_{\mathrm{segment}}=250\text{ m}}
\]

\[
\boxed{r_{\mathrm{raster}}=25\text{ m}}
\]

with primary temporal-corroboration settings:

\[
\boxed{T=50\text{ m},\qquad G_{\max}=48\text{ h}}
\]

where:

- \(T\) is the spatial challenge tolerance;
- \(G_{\max}\) is the maximum NIROPS observation gap accepted for a resolved transition.

---

# Production objectives

For every exact Fireline–NIROPS matched incident containing `Held` or `Burned Over` canonical firelines:

1. segment parent lines into local 250 m modeling units;
2. preserve parent-line provenance;
3. build segment representative points;
4. reconstruct signed NIROPS distance trajectories;
5. use one incident-local raster when memory-safe;
6. automatically switch to tiled rasters for geographically large incidents;
7. save the trajectory matrix before applying temporal thresholds;
8. generate a row-level primary 50 m / 48 h temporal classification;
9. generate sensitivity summaries over:

\[
T\in\{25,50,100,250\}\text{ m}
\]

and

\[
G\in\{24,48,72\}\text{ h};
\]

10. save a temporally supported subset for later feature extraction and ML.

---

# Important scientific interpretation

The original Fireline `Held` / `Burned Over` outcome is a quality-assured spatially inferred outcome based mainly on final-perimeter geometry.

The NIROPS progression is used here to reconstruct whether observed fire progression temporally corroborates the inherited segment outcome.

A clean crossing does **not** provide an exact failure time.

For a supported Burned Over segment:

\[
t_{\mathrm{challenge}}
\in
[t_j,t_{j+1}]
\]

where NIROPS shows the segment outside the observed burned area at \(t_j\) and inside at \(t_{j+1}\).

The same interval-censoring principle is retained for supported Held challenges.

---

# Production architecture

The row hierarchy is:

\[
\text{Incident}
\rightarrow
\text{Parent fireline}
\rightarrow
\text{250 m segment}
\rightarrow
\text{NIROPS signed-distance trajectory}.
\]

The notebook intentionally avoids creating one giant in-memory dataset.

Each incident is processed independently and written as a shard.

This makes the production run:

- resumable;
- inspectable;
- less memory intensive;
- safer if one incident fails.



> **03G_v2 fixes**
>
> 1. Fixes the Pandas `itertuples()` bug for the underscore-prefixed NIROPS `_time` field by using `name=None`.
> 2. Reuses existing `segments/<IRWIN>.parquet` shards from the failed 03G_v1 run instead of segmenting the 415 incidents again.
> 3. Accepts the existing v1 production configuration when all scientific settings are identical.
> 4. Writes a separate `PRODUCTION_RUN_LOG_V2.csv`.
> 5. Preserves the existing resumable per-incident production architecture.


In [ ]:
import os  # Configurable project root for the public release.

NOTEBOOK_BUILD_ID = "03G_TEMPORAL_PRODUCTION_V2"

print("=" * 78)
print("RUNNING:", NOTEBOOK_BUILD_ID)
print("=" * 78)

from pathlib import Path

PROJECT_ROOT = Path(os.environ.get("FIRELINE_PROJECT_ROOT", "."))

FIRELINE_PARQUET = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "fireline"
    / "fireline_qaqc_version_resolved.parquet"
)

NIROPS_SEARCH_ROOT = (
    PROJECT_ROOT
    / "data"
    / "interim"
    / "joinability_audit"
    / "nirops_extracted"
)

PROCESSED_ROOT = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "fireline_dynamic_250m"
)

OUTPUT_ROOT = (
    PROJECT_ROOT
    / "outputs"
    / "nirops_temporal_production_250m"
)

SEGMENT_ROOT = (
    PROCESSED_ROOT
    / "segments"
)

TRAJECTORY_ROOT = (
    PROCESSED_ROOT
    / "trajectories"
)

PRIMARY_ROOT = (
    PROCESSED_ROOT
    / "primary_50m_48h"
)

SUPPORTED_ROOT = (
    PROCESSED_ROOT
    / "temporally_supported_50m_48h"
)

SENSITIVITY_ROOT = (
    PROCESSED_ROOT
    / "sensitivity"
)

INCIDENT_META_ROOT = (
    PROCESSED_ROOT
    / "incident_metadata"
)

for directory in [
    PROCESSED_ROOT,
    OUTPUT_ROOT,
    SEGMENT_ROOT,
    TRAJECTORY_ROOT,
    PRIMARY_ROOT,
    SUPPORTED_ROOT,
    SENSITIVITY_ROOT,
    INCIDENT_META_ROOT,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )

assert FIRELINE_PARQUET.exists(), FIRELINE_PARQUET
assert NIROPS_SEARCH_ROOT.exists(), NIROPS_SEARCH_ROOT

# ============================================================
# PRODUCTION SETTINGS
# ============================================================

SEGMENT_LENGTH_M = 250.0

RASTER_RESOLUTION_M = 25.0

PRIMARY_TOLERANCE_M = 50.0
PRIMARY_MAX_GAP_HOURS = 48.0

SPATIAL_TOLERANCES_M = [
    25.0,
    50.0,
    100.0,
    250.0,
]

TEMPORAL_GAPS_HOURS = [
    24.0,
    48.0,
    72.0,
]

MAX_DISTANCE_M = max(
    SPATIAL_TOLERANCES_M
)

FAR_DISTANCE_M = (
    MAX_DISTANCE_M
    + 1.0
)

GRID_PADDING_M = (
    MAX_DISTANCE_M
    + 2 * RASTER_RESOLUTION_M
)

# Use the efficient one-grid architecture whenever possible.
MAX_SINGLE_GRID_CELLS = 20_000_000

# If the incident exceeds the single-grid threshold, points are
# divided into fixed raster tiles.
TILE_CORE_PIXELS = 2000

TILE_CORE_SIZE_M = (
    TILE_CORE_PIXELS
    * RASTER_RESOLUTION_M
)

TILE_HALO_M = (
    MAX_DISTANCE_M
    + 2 * RASTER_RESOLUTION_M
)

MIN_OBSERVATIONS = 3

ENGAGED_LABELS = [
    "Held",
    "Burned Over",
]

METRIC_CRS = "EPSG:5070"

USE_CUMULATIVE_NIROPS = False

# Resume an interrupted production run.
RESUME = True

# None = process all eligible exact-matched incidents.
MAX_INCIDENTS = None

# Small/medium incidents first. Large incidents then run later,
# while all completed shards remain safely on disk.
PROCESS_SMALLEST_FIRST = True

print("Primary segment length:", SEGMENT_LENGTH_M, "m")
print("Raster resolution:", RASTER_RESOLUTION_M, "m")
print("Primary tolerance:", PRIMARY_TOLERANCE_M, "m")
print("Primary max gap:", PRIMARY_MAX_GAP_HOURS, "h")
print("Single-grid cell limit:", f"{MAX_SINGLE_GRID_CELLS:,}")
print("Tile core:", TILE_CORE_SIZE_M / 1000, "km")
print("Resume:", RESUME)
print("MAX_INCIDENTS:", MAX_INCIDENTS)


## 1. Imports

In [ ]:

import sys
import subprocess
import importlib.util

required = {
    "pandas": "pandas",
    "geopandas": "geopandas",
    "pyogrio": "pyogrio",
    "shapely": "shapely",
    "pyarrow": "pyarrow",
    "tqdm": "tqdm",
    "rasterio": "rasterio",
    "scipy": "scipy",
}

for module, package in required.items():

    if importlib.util.find_spec(
        module
    ) is None:

        subprocess.check_call(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                package,
            ]
        )

import re
import gc
import json
import time
import traceback
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import geopandas as gpd
import shapely

from tqdm.auto import tqdm

from rasterio.features import rasterize
from rasterio.transform import from_origin

from scipy.ndimage import distance_transform_edt

pd.set_option("display.max_columns", 250)
pd.set_option("display.max_rows", 250)
pd.set_option("display.width", 200)

print("Python:", sys.executable)
print("GeoPandas:", gpd.__version__)
print("Shapely:", shapely.__version__)

assert int(
    shapely.__version__.split(".")[0]
) >= 2



# 2. Freeze the production configuration

A configuration file prevents accidentally mixing shards created under different scientific settings.

If the output folder already contains a configuration from another build or another segment/raster setting, the notebook stops instead of silently combining incompatible data.


In [ ]:

SCIENTIFIC_CONFIG = {
    "segment_length_m": SEGMENT_LENGTH_M,
    "raster_resolution_m": RASTER_RESOLUTION_M,
    "primary_tolerance_m": PRIMARY_TOLERANCE_M,
    "primary_max_gap_hours": PRIMARY_MAX_GAP_HOURS,
    "spatial_tolerances_m": SPATIAL_TOLERANCES_M,
    "temporal_gaps_hours": TEMPORAL_GAPS_HOURS,
    "max_distance_m": MAX_DISTANCE_M,
    "max_single_grid_cells": MAX_SINGLE_GRID_CELLS,
    "tile_core_pixels": TILE_CORE_PIXELS,
    "tile_core_size_m": TILE_CORE_SIZE_M,
    "tile_halo_m": TILE_HALO_M,
    "metric_crs": METRIC_CRS,
    "use_cumulative_nirops": USE_CUMULATIVE_NIROPS,
    "engaged_labels": ENGAGED_LABELS,
}

PRODUCTION_CONFIG = {
    "notebook_build_id": NOTEBOOK_BUILD_ID,
    **SCIENTIFIC_CONFIG,
}

CONFIG_PATH = (
    PROCESSED_ROOT
    / "PRODUCTION_CONFIG.json"
)

if CONFIG_PATH.exists():

    existing_config = json.loads(
        CONFIG_PATH.read_text(
            encoding="utf-8"
        )
    )

    # V2 intentionally ignores only the old notebook build ID.
    # All scientific settings must still match exactly.
    existing_scientific = {
        key: value
        for key, value in existing_config.items()
        if key != "notebook_build_id"
    }

    if existing_scientific != SCIENTIFIC_CONFIG:

        raise RuntimeError(
            "Existing production configuration has different scientific "
            "settings. Do NOT mix the shards. Use a new output directory "
            "or intentionally remove the incompatible production folder."
        )

    old_build = existing_config.get(
        "notebook_build_id",
        "UNKNOWN"
    )

    print(
        "Existing compatible production configuration found."
    )
    print(
        "Previous notebook build:",
        old_build
    )
    print(
        "Current notebook build:",
        NOTEBOOK_BUILD_ID
    )
    print(
        "Reusing compatible v1 segment shards and any complete v2-compatible shards."
    )

    # Update only the build provenance; scientific settings are unchanged.
    CONFIG_PATH.write_text(
        json.dumps(
            PRODUCTION_CONFIG,
            indent=2,
        ),
        encoding="utf-8",
    )

else:

    CONFIG_PATH.write_text(
        json.dumps(
            PRODUCTION_CONFIG,
            indent=2,
        ),
        encoding="utf-8",
    )

print(
    json.dumps(
        PRODUCTION_CONFIG,
        indent=2,
    )
)


## 3. Load canonical Fireline and NIROPS

In [ ]:

load_start = time.perf_counter()

fireline = gpd.read_parquet(
    FIRELINE_PARQUET
)

nirops_candidates = list(
    NIROPS_SEARCH_ROOT.rglob(
        "NIROPS_2020_2024_R1_R6.shp"
    )
)

if not nirops_candidates:

    nirops_candidates = list(
        NIROPS_SEARCH_ROOT.rglob(
            "*.shp"
        )
    )

assert nirops_candidates, (
    "NIROPS shapefile not found."
)

NIROPS_SHP = (
    nirops_candidates[0]
)

nirops = gpd.read_file(
    NIROPS_SHP,
    engine="pyogrio",
)

print(
    "Load time:",
    f"{time.perf_counter() - load_start:.2f} sec"
)

print(
    "Fireline rows:",
    f"{len(fireline):,}"
)

print(
    "NIROPS rows:",
    f"{len(nirops):,}"
)

print(
    "Fireline CRS:",
    fireline.crs
)

print(
    "NIROPS CRS:",
    nirops.crs
)


## 4. Detect and normalize NIROPS IRWIN/time fields

In [ ]:

def normalize_column_name(
    value,
):
    return re.sub(
        r"[^a-z0-9]",
        "",
        str(
            value
        ).lower(),
    )


def find_column(
    columns,
    exact=(),
    contains=(),
):

    lookup = {
        normalize_column_name(
            column
        ): column
        for column in columns
    }

    for candidate in exact:

        key = normalize_column_name(
            candidate
        )

        if key in lookup:
            return lookup[
                key
            ]

    for token in contains:

        token = normalize_column_name(
            token
        )

        for key, original in (
            lookup.items()
        ):

            if token in key:
                return original

    return None


def normalize_irwin(
    value,
):

    if pd.isna(
        value
    ):
        return None

    value = re.sub(
        r"[^A-Z0-9]",
        "",
        str(
            value
        ).strip().upper(),
    )

    return (
        value
        or None
    )


NR_IRWIN = find_column(
    nirops.columns,
    exact=[
        "IRWINID",
        "IRWIN_ID",
        "IrwinID",
    ],
    contains=[
        "irwin",
    ],
)

NR_TIME = find_column(
    nirops.columns,
    exact=[
        "UTC",
        "DateUTC",
        "DateTimeUTC",
    ],
    contains=[
        "utc",
        "dateutc",
    ],
)

assert NR_IRWIN is not None
assert NR_TIME is not None

nirops[
    "_irwin_norm"
] = (
    nirops[
        NR_IRWIN
    ]
    .map(
        normalize_irwin
    )
)

nirops[
    "_time"
] = pd.to_datetime(
    nirops[
        NR_TIME
    ],
    errors="coerce",
    utc=True,
)

print(
    "NIROPS IRWIN field:",
    NR_IRWIN
)

print(
    "NIROPS time field:",
    NR_TIME
)


## 5. Build the exact matched production inventory

In [ ]:

fire_ids = set(
    fireline[
        "_irwin_norm"
    ]
    .dropna()
    .unique()
)

nirops_ids = set(
    nirops[
        "_irwin_norm"
    ]
    .dropna()
    .unique()
)

matched_ids = (
    fire_ids
    & nirops_ids
)

matched_fireline = (
    fireline[
        fireline[
            "_irwin_norm"
        ].isin(
            matched_ids
        )
    ]
    .copy()
)

matched_nirops = (
    nirops[
        nirops[
            "_irwin_norm"
        ].isin(
            matched_ids
        )
    ]
    .copy()
)

engaged_fireline = (
    matched_fireline[
        matched_fireline[
            "_engagement_norm"
        ].isin(
            ENGAGED_LABELS
        )
    ]
    .copy()
)

fireline_counts = (
    engaged_fireline
    .groupby(
        "_irwin_norm"
    )
    .agg(
        engaged_parent_rows=(
            "_irwin_norm",
            "size",
        ),
        held_parent_rows=(
            "_engagement_norm",
            lambda s: int(
                (
                    s
                    == "Held"
                ).sum()
            ),
        ),
        burned_over_parent_rows=(
            "_engagement_norm",
            lambda s: int(
                (
                    s
                    == "Burned Over"
                ).sum()
            ),
        ),
        incident_name=(
            "IncidentName",
            "first",
        ),
        year=(
            "Year",
            "first",
        ),
        engaged_length_km=(
            "_length_km_geom",
            "sum",
        ),
    )
    .reset_index()
)

nirops_counts = (
    matched_nirops
    .groupby(
        "_irwin_norm"
    )
    .agg(
        nirops_rows=(
            "_irwin_norm",
            "size",
        ),
        nirops_times=(
            "_time",
            lambda values: (
                values
                .dropna()
                .nunique()
            ),
        ),
        first_nirops_time=(
            "_time",
            "min",
        ),
        last_nirops_time=(
            "_time",
            "max",
        ),
    )
    .reset_index()
)

incident_inventory = (
    fireline_counts
    .merge(
        nirops_counts,
        on="_irwin_norm",
        how="inner",
    )
)

incident_inventory = (
    incident_inventory[
        incident_inventory[
            "nirops_times"
        ] >= MIN_OBSERVATIONS
    ]
    .copy()
)

incident_inventory[
    "estimated_work"
] = (
    incident_inventory[
        "engaged_parent_rows"
    ]
    * incident_inventory[
        "nirops_times"
    ]
)

if PROCESS_SMALLEST_FIRST:

    incident_inventory = (
        incident_inventory
        .sort_values(
            [
                "estimated_work",
                "engaged_parent_rows",
            ],
            ascending=True,
        )
        .reset_index(
            drop=True
        )
    )

else:

    incident_inventory = (
        incident_inventory
        .sort_values(
            [
                "year",
                "incident_name",
            ]
        )
        .reset_index(
            drop=True
        )
    )

if MAX_INCIDENTS is not None:

    incident_inventory = (
        incident_inventory
        .head(
            int(
                MAX_INCIDENTS
            )
        )
        .copy()
    )

PRODUCTION_IDS = (
    incident_inventory[
        "_irwin_norm"
    ]
    .tolist()
)

print("=" * 78)
print(
    "EXACT MATCHED IDS:",
    f"{len(matched_ids):,}"
)
print(
    "PRODUCTION-ELIGIBLE INCIDENTS:",
    f"{len(PRODUCTION_IDS):,}"
)
print(
    "PROGRESS BAR SHOULD END AT:",
    len(PRODUCTION_IDS)
)
print("=" * 78)

display(
    incident_inventory.head(
        20
    )
)

incident_inventory.to_csv(
    OUTPUT_ROOT
    / "production_incident_inventory.csv",
    index=False,
)



# 6. Project NIROPS once

The NIROPS dataset is small enough to project the exact matched observations once.

Fireline geometry is projected incident-by-incident so the production run does not hold the entire canonical Fireline geometry in two CRSs simultaneously.


In [ ]:

nirops_projection_start = (
    time.perf_counter()
)

matched_nirops_metric = (
    matched_nirops
    .to_crs(
        METRIC_CRS
    )
)

print(
    "Projected matched NIROPS rows:",
    f"{len(matched_nirops_metric):,}"
)

print(
    "Projection time:",
    f"{time.perf_counter() - nirops_projection_start:.2f} sec"
)



# 7. Segment one incident into 250 m modeling units

The production representation stores the segment midpoint and segment metadata rather than materializing millions of segment `LineString` geometries.

Each segment retains:

- parent source index;
- parent geometry hash;
- parent engagement outcome;
- feature category;
- component index;
- segment index;
- start/end distance along the parent component;
- actual segment length;
- midpoint;
- chord orientation.

The local segment geometry can be reconstructed from the parent line later if required.


In [ ]:

SEGMENT_PARENT_COLUMNS = [
    "source_index",
    "_irwin_norm",
    "IncidentName",
    "Year",
    "FeatureCategory",
    "_engagement_norm",
    "_geom_hash",
    "_length_km_geom",
]


def line_components(
    geometry,
):

    if (
        geometry is None
        or geometry.is_empty
    ):
        return []

    if (
        geometry.geom_type
        == "LineString"
    ):
        return [
            geometry
        ]

    if (
        geometry.geom_type
        == "MultiLineString"
    ):
        return list(
            geometry.geoms
        )

    return []


def segment_incident(
    incident_lines,
):

    incident_lines_metric = (
        incident_lines
        .to_crs(
            METRIC_CRS
        )
        .reset_index()
        .rename(
            columns={
                "index": "source_index"
            }
        )
    )

    data_columns = (
        SEGMENT_PARENT_COLUMNS
        + [
            "geometry",
        ]
    )

    frames = []

    tuple_rows = (
        incident_lines_metric[
            data_columns
        ]
        .itertuples(
            index=False,
            name=None,
        )
    )

    for values in tuple_rows:

        row = dict(
            zip(
                data_columns,
                values,
            )
        )

        parent_geometry = (
            row[
                "geometry"
            ]
        )

        components = (
            line_components(
                parent_geometry
            )
        )

        for (
            component_index,
            line,
        ) in enumerate(
            components
        ):

            component_length = float(
                line.length
            )

            if (
                not np.isfinite(
                    component_length
                )
                or component_length <= 0
            ):
                continue

            n_segments = max(
                1,
                int(
                    np.ceil(
                        component_length
                        / SEGMENT_LENGTH_M
                    )
                ),
            )

            starts = (
                np.arange(
                    n_segments,
                    dtype=np.float64,
                )
                * SEGMENT_LENGTH_M
            )

            ends = np.minimum(
                starts
                + SEGMENT_LENGTH_M,
                component_length,
            )

            mids = (
                starts
                + ends
            ) / 2.0

            midpoints = (
                shapely.line_interpolate_point(
                    line,
                    mids,
                )
            )

            start_points = (
                shapely.line_interpolate_point(
                    line,
                    starts,
                )
            )

            end_points = (
                shapely.line_interpolate_point(
                    line,
                    ends,
                )
            )

            mid_x = (
                shapely.get_x(
                    midpoints
                )
            )

            mid_y = (
                shapely.get_y(
                    midpoints
                )
            )

            start_x = (
                shapely.get_x(
                    start_points
                )
            )

            start_y = (
                shapely.get_y(
                    start_points
                )
            )

            end_x = (
                shapely.get_x(
                    end_points
                )
            )

            end_y = (
                shapely.get_y(
                    end_points
                )
            )

            dx = (
                end_x
                - start_x
            )

            dy = (
                end_y
                - start_y
            )

            # 0 = north, 90 = east.
            # Line orientation is bidirectional, hence modulo 180.
            orientation_deg = (
                np.degrees(
                    np.arctan2(
                        dx,
                        dy,
                    )
                )
                % 180.0
            )

            frame = pd.DataFrame(
                {
                    "source_index": row[
                        "source_index"
                    ],
                    "_irwin_norm": row[
                        "_irwin_norm"
                    ],
                    "IncidentName": row[
                        "IncidentName"
                    ],
                    "Year": row[
                        "Year"
                    ],
                    "FeatureCategory": row[
                        "FeatureCategory"
                    ],
                    "_engagement_norm": row[
                        "_engagement_norm"
                    ],
                    "_geom_hash": row[
                        "_geom_hash"
                    ],
                    "_length_km_geom": row[
                        "_length_km_geom"
                    ],
                    "component_index": component_index,
                    "segment_index": np.arange(
                        n_segments,
                        dtype=np.int32,
                    ),
                    "segment_start_m": starts,
                    "segment_end_m": ends,
                    "actual_segment_length_m": (
                        ends
                        - starts
                    ),
                    "segment_mid_x_m": mid_x,
                    "segment_mid_y_m": mid_y,
                    "segment_orientation_deg_0_180": orientation_deg,
                }
            )

            frame[
                "geometry"
            ] = list(
                midpoints
            )

            frames.append(
                frame
            )

    if not frames:

        return gpd.GeoDataFrame(
            columns=[
                *SEGMENT_PARENT_COLUMNS,
                "component_index",
                "segment_index",
                "segment_start_m",
                "segment_end_m",
                "actual_segment_length_m",
                "segment_mid_x_m",
                "segment_mid_y_m",
                "segment_orientation_deg_0_180",
                "segment_uid",
                "segment_row_index",
                "geometry",
            ],
            geometry="geometry",
            crs=METRIC_CRS,
        )

    segments = gpd.GeoDataFrame(
        pd.concat(
            frames,
            ignore_index=True,
        ),
        geometry="geometry",
        crs=METRIC_CRS,
    )

    segments[
        "segment_uid"
    ] = (
        segments[
            "_irwin_norm"
        ].astype(str)
        + "_"
        + segments[
            "_geom_hash"
        ].astype(str)
        + "_c"
        + segments[
            "component_index"
        ].astype(str)
        + "_s"
        + segments[
            "segment_index"
        ].astype(str)
    )

    segments[
        "segment_row_index"
    ] = np.arange(
        len(
            segments
        ),
        dtype=np.int64,
    )

    return (
        segments
    )


## 8. NIROPS progression helper

In [ ]:

def build_progression(
    incident_nirops,
):

    progression = (
        incident_nirops
        .dropna(
            subset=[
                "_time",
            ]
        )[
            [
                "_time",
                "geometry",
            ]
        ]
        .sort_values(
            "_time"
        )
        .copy()
    )

    if progression.empty:

        return progression

    if not progression[
        "_time"
    ].is_unique:

        rows = []

        for (
            timestamp,
            group,
        ) in progression.groupby(
            "_time"
        ):

            geometry = (
                group.geometry.iloc[0]
                if len(
                    group
                ) == 1
                else group.geometry.union_all()
            )

            rows.append(
                {
                    "_time": timestamp,
                    "geometry": geometry,
                }
            )

        progression = (
            gpd.GeoDataFrame(
                rows,
                geometry="geometry",
                crs=incident_nirops.crs,
            )
        )

    if USE_CUMULATIVE_NIROPS:

        running = None
        cumulative = []

        for geometry in (
            progression.geometry
        ):

            running = (
                geometry
                if running is None
                else running.union(
                    geometry
                )
            )

            cumulative.append(
                running
            )

        progression = (
            progression.copy()
        )

        progression[
            "geometry"
        ] = cumulative

    return (
        progression
        .sort_values(
            "_time"
        )
        .reset_index(
            drop=True
        )
    )



# 9. Raster region helpers

The processor is adaptive.

### Normal incident

If the incident-wide 25 m raster is at most 20 million cells:

\[
\text{one incident}\rightarrow\text{one raster region}.
\]

This preserves the efficient architecture demonstrated by Notebook 03E.

### Large geographic incident

If the incident-wide raster would exceed the memory threshold:

\[
\text{segment points}
\rightarrow
50\text{ km core tiles}
\rightarrow
300\text{ m halo}.
\]

The halo exceeds the maximum 250 m scientific distance threshold.

Therefore any true perimeter boundary within 250 m of a core point must be represented inside that point's raster tile.

Distances larger than 250 m are intentionally capped at 251 m.


In [ ]:

def bounds_intersect(
    a,
    b,
):
    """
    Axis-aligned bounding-box intersection.

    bounds = (minx, miny, maxx, maxy)
    """

    return not (
        a[2] < b[0]
        or a[0] > b[2]
        or a[3] < b[1]
        or a[1] > b[3]
    )


def make_region(
    point_indices,
    x,
    y,
    minx,
    miny,
    maxx,
    maxy,
    mode,
    region_id,
):

    resolution = (
        RASTER_RESOLUTION_M
    )

    width = int(
        np.ceil(
            (
                maxx
                - minx
            )
            / resolution
        )
    )

    height = int(
        np.ceil(
            (
                maxy
                - miny
            )
            / resolution
        )
    )

    transform = (
        from_origin(
            minx,
            maxy,
            resolution,
            resolution,
        )
    )

    local_x = (
        x[
            point_indices
        ]
    )

    local_y = (
        y[
            point_indices
        ]
    )

    point_cols = np.floor(
        (
            local_x
            - minx
        )
        / resolution
    ).astype(
        np.int64
    )

    point_rows = np.floor(
        (
            maxy
            - local_y
        )
        / resolution
    ).astype(
        np.int64
    )

    point_cols = np.clip(
        point_cols,
        0,
        width - 1,
    )

    point_rows = np.clip(
        point_rows,
        0,
        height - 1,
    )

    return {
        "mode": mode,
        "region_id": region_id,
        "point_indices": np.asarray(
            point_indices,
            dtype=np.int64,
        ),
        "point_rows": point_rows,
        "point_cols": point_cols,
        "width": width,
        "height": height,
        "cells": (
            width
            * height
        ),
        "transform": transform,
        "bounds": (
            float(
                minx
            ),
            float(
                miny
            ),
            float(
                maxx
            ),
            float(
                maxy
            ),
        ),
    }


def build_processing_regions(
    segment_points,
):

    x = (
        segment_points.geometry.x
        .to_numpy(
            dtype=np.float64
        )
    )

    y = (
        segment_points.geometry.y
        .to_numpy(
            dtype=np.float64
        )
    )

    resolution = (
        RASTER_RESOLUTION_M
    )

    # --------------------------------------------------------
    # Test one incident-wide raster first.
    # --------------------------------------------------------

    single_minx = (
        np.floor(
            (
                x.min()
                - GRID_PADDING_M
            )
            / resolution
        )
        * resolution
    )

    single_maxx = (
        np.ceil(
            (
                x.max()
                + GRID_PADDING_M
            )
            / resolution
        )
        * resolution
    )

    single_miny = (
        np.floor(
            (
                y.min()
                - GRID_PADDING_M
            )
            / resolution
        )
        * resolution
    )

    single_maxy = (
        np.ceil(
            (
                y.max()
                + GRID_PADDING_M
            )
            / resolution
        )
        * resolution
    )

    single_width = int(
        np.ceil(
            (
                single_maxx
                - single_minx
            )
            / resolution
        )
    )

    single_height = int(
        np.ceil(
            (
                single_maxy
                - single_miny
            )
            / resolution
        )
    )

    single_cells = (
        single_width
        * single_height
    )

    if (
        single_cells
        <= MAX_SINGLE_GRID_CELLS
    ):

        region = make_region(
            point_indices=np.arange(
                len(
                    segment_points
                )
            ),
            x=x,
            y=y,
            minx=single_minx,
            miny=single_miny,
            maxx=single_maxx,
            maxy=single_maxy,
            mode="single",
            region_id="single",
        )

        return {
            "mode": "single",
            "single_grid_cells": single_cells,
            "regions": [
                region,
            ],
        }

    # --------------------------------------------------------
    # Tiled mode.
    # --------------------------------------------------------

    tile_x = np.floor(
        x
        / TILE_CORE_SIZE_M
    ).astype(
        np.int64
    )

    tile_y = np.floor(
        y
        / TILE_CORE_SIZE_M
    ).astype(
        np.int64
    )

    tile_pairs = np.column_stack(
        [
            tile_x,
            tile_y,
        ]
    )

    unique_tiles = np.unique(
        tile_pairs,
        axis=0,
    )

    regions = []

    for tile_pair in unique_tiles:

        tx = int(
            tile_pair[0]
        )

        ty = int(
            tile_pair[1]
        )

        point_indices = np.flatnonzero(
            (
                tile_x
                == tx
            )
            & (
                tile_y
                == ty
            )
        )

        core_minx = (
            tx
            * TILE_CORE_SIZE_M
        )

        core_maxx = (
            core_minx
            + TILE_CORE_SIZE_M
        )

        core_miny = (
            ty
            * TILE_CORE_SIZE_M
        )

        core_maxy = (
            core_miny
            + TILE_CORE_SIZE_M
        )

        region = make_region(
            point_indices=point_indices,
            x=x,
            y=y,
            minx=(
                core_minx
                - TILE_HALO_M
            ),
            miny=(
                core_miny
                - TILE_HALO_M
            ),
            maxx=(
                core_maxx
                + TILE_HALO_M
            ),
            maxy=(
                core_maxy
                + TILE_HALO_M
            ),
            mode="tile",
            region_id=(
                f"tile_{tx}_{ty}"
            ),
        )

        regions.append(
            region
        )

    return {
        "mode": "tiled",
        "single_grid_cells": single_cells,
        "regions": regions,
    }



# 10. Rasterize one observation within one processing region

Important edge cases are handled explicitly.

### Mask contains no burned pixels

All segment points assigned to the region are farther than the useful distance threshold:

\[
s=+251\text{ m}.
\]

### Mask is completely burned

No observed perimeter boundary is present within the tile halo:

\[
s=-251\text{ m}.
\]

### Mixed burned/unburned mask

Euclidean distance transforms provide the capped signed distance.


In [ ]:

def sample_region_signed_distance(
    polygon,
    polygon_bounds,
    region,
):

    # If even the polygon bounding box misses the halo region,
    # every core point is safely farther than MAX_DISTANCE_M.
    if not bounds_intersect(
        polygon_bounds,
        region[
            "bounds"
        ],
    ):

        return np.full(
            len(
                region[
                    "point_indices"
                ]
            ),
            FAR_DISTANCE_M,
            dtype=np.float32,
        )

    mask = rasterize(
        [
            (
                polygon,
                1,
            )
        ],
        out_shape=(
            region[
                "height"
            ],
            region[
                "width"
            ],
        ),
        transform=region[
            "transform"
        ],
        fill=0,
        default_value=1,
        dtype="uint8",
        all_touched=False,
    )

    rows = (
        region[
            "point_rows"
        ]
    )

    cols = (
        region[
            "point_cols"
        ]
    )

    # No observed burned area within the halo raster.
    if not mask.any():

        return np.full(
            len(
                rows
            ),
            FAR_DISTANCE_M,
            dtype=np.float32,
        )

    # Entire halo raster is observed burned area.
    if mask.all():

        return np.full(
            len(
                rows
            ),
            -FAR_DISTANCE_M,
            dtype=np.float32,
        )

    point_inside = (
        mask[
            rows,
            cols
        ]
        == 1
    )

    signed = np.full(
        len(
            rows
        ),
        FAR_DISTANCE_M,
        dtype=np.float32,
    )

    if point_inside.any():

        inside_distance = (
            distance_transform_edt(
                mask,
                sampling=RASTER_RESOLUTION_M,
            )
        )

        values = (
            inside_distance[
                rows[
                    point_inside
                ],
                cols[
                    point_inside
                ],
            ]
        )

        signed[
            point_inside
        ] = (
            -np.minimum(
                values,
                FAR_DISTANCE_M,
            )
        ).astype(
            np.float32
        )

        del inside_distance

    point_outside = (
        ~point_inside
    )

    if point_outside.any():

        outside_distance = (
            distance_transform_edt(
                mask == 0,
                sampling=RASTER_RESOLUTION_M,
            )
        )

        values = (
            outside_distance[
                rows[
                    point_outside
                ],
                cols[
                    point_outside
                ],
            ]
        )

        signed[
            point_outside
        ] = (
            np.minimum(
                values,
                FAR_DISTANCE_M,
            )
        ).astype(
            np.float32
        )

        del outside_distance

    del mask

    return signed



# 11. Build an incident trajectory matrix

The matrix has shape:

\[
N_{\mathrm{segments}}
\times
N_{\mathrm{NIROPS\ times}}.
\]

Each row is aligned with `segment_row_index` in the incident segment Parquet shard.

Each column is aligned with the timestamp stored in the trajectory NPZ.

A nested progress bar provides a visible heartbeat while a large incident is running.


In [ ]:

def build_incident_trajectory(
    segment_points,
    incident_nirops,
    incident_label,
):

    progression = (
        build_progression(
            incident_nirops
        )
    )

    if len(
        progression
    ) < MIN_OBSERVATIONS:

        return {
            "status": "TOO_FEW_OBSERVATIONS",
        }

    region_info = (
        build_processing_regions(
            segment_points
        )
    )

    regions = (
        region_info[
            "regions"
        ]
    )

    signed = np.full(
        (
            len(
                segment_points
            ),
            len(
                progression
            ),
        ),
        FAR_DISTANCE_M,
        dtype=np.float32,
    )

    time_runtime_rows = []

    # IMPORTANT V2 FIX:
    # Use name=None so underscore-prefixed "_time" is not renamed by
    # pandas namedtuple machinery.
    inner = tqdm(
        progression[
            [
                "_time",
                "geometry",
            ]
        ].itertuples(
            index=False,
            name=None,
        ),
        total=len(
            progression
        ),
        desc=(
            f"{incident_label[:24]} NIROPS"
        ),
        leave=False,
    )

    for (
        time_index,
        (
            timestamp,
            polygon,
        ),
    ) in enumerate(
        inner
    ):

        observation_start = (
            time.perf_counter()
        )

        polygon_bounds = (
            polygon.bounds
        )

        for region in regions:

            local_signed = (
                sample_region_signed_distance(
                    polygon=polygon,
                    polygon_bounds=polygon_bounds,
                    region=region,
                )
            )

            signed[
                region[
                    "point_indices"
                ],
                time_index
            ] = local_signed

        total_seconds = (
            time.perf_counter()
            - observation_start
        )

        time_runtime_rows.append(
            {
                "observation_index": time_index,
                "timestamp": timestamp,
                "region_mode": region_info[
                    "mode"
                ],
                "regions": len(
                    regions
                ),
                "observation_seconds": total_seconds,
            }
        )

        inner.set_postfix(
            {
                "regions": len(
                    regions
                ),
                "sec": f"{total_seconds:.2f}",
            }
        )

        gc.collect()

    return {
        "status": "OK",
        "signed": signed,
        "times": pd.DatetimeIndex(
            progression[
                "_time"
            ]
        ),
        "region_info": region_info,
        "observation_runtime": pd.DataFrame(
            time_runtime_rows
        ),
    }



# 12. Detailed interval-censored temporal classifier

This classifier is threshold-based and therefore can be rerun cheaply from the cached trajectory matrix.

## Burned Over support

A supported Burned Over segment requires:

\[
\text{outside at }t_j
\rightarrow
\text{inside at }t_{j+1}
\]

with:

\[
t_{j+1}-t_j\le G.
\]

## Held support

A supported Held segment requires:

\[
\text{outside at }t_j
\rightarrow
\text{near at }t_{j+1}
\]

with:

- gap \(\le G\);
- at least one later NIROPS observation;
- no later observation placing the midpoint clearly inside the burned area.

The resulting challenge time remains interval-censored.


In [ ]:

def first_true_index(
    mask,
):

    result = np.full(
        mask.shape[0],
        -1,
        dtype=np.int32,
    )

    if (
        mask.ndim != 2
        or mask.shape[1] == 0
    ):
        return result

    has_true = (
        mask.any(
            axis=1
        )
    )

    result[
        has_true
    ] = (
        mask[
            has_true
        ]
        .argmax(
            axis=1
        )
    )

    return result


def last_true_index(
    mask,
):

    result = np.full(
        mask.shape[0],
        -1,
        dtype=np.int32,
    )

    if (
        mask.ndim != 2
        or mask.shape[1] == 0
    ):
        return result

    has_true = (
        mask.any(
            axis=1
        )
    )

    reversed_index = (
        mask[
            has_true,
            ::-1
        ]
        .argmax(
            axis=1
        )
    )

    result[
        has_true
    ] = (
        mask.shape[1]
        - 1
        - reversed_index
    )

    return result


def classify_temporal(
    labels,
    signed,
    times,
    tolerance_m,
    max_gap_hours,
    return_event_times=True,
):

    labels = np.asarray(
        labels,
        dtype=object,
    )

    n_segments, n_times = (
        signed.shape
    )

    if n_times < 2:

        raise ValueError(
            "At least two NIROPS times are required."
        )

    near = (
        np.abs(
            signed
        )
        <= tolerance_m
    )

    outside = (
        signed
        > tolerance_m
    )

    inside = (
        signed
        < -tolerance_m
    )

    gaps_hours = np.array(
        [
            (
                times[j + 1]
                - times[j]
            ).total_seconds()
            / 3600.0
            for j in range(
                n_times - 1
            )
        ],
        dtype=np.float32,
    )

    short_gap = (
        gaps_hours
        <= max_gap_hours
    )

    long_gap = (
        gaps_hours
        > max_gap_hours
    )

    cross_any = (
        outside[
            :,
            :-1
        ]
        & inside[
            :,
            1:
        ]
    )

    cross_short = (
        cross_any
        & short_gap[
            np.newaxis,
            :
        ]
    )

    cross_long = (
        cross_any
        & long_gap[
            np.newaxis,
            :
        ]
    )

    # Held support requires outside -> near, not outside -> inside.
    approach_near_any = (
        outside[
            :,
            :-1
        ]
        & near[
            :,
            1:
        ]
    )

    approach_near_short = (
        approach_near_any
        & short_gap[
            np.newaxis,
            :
        ]
    )

    approach_near_long = (
        approach_near_any
        & long_gap[
            np.newaxis,
            :
        ]
    )

    first_cross_short = (
        first_true_index(
            cross_short
        )
    )

    first_cross_long = (
        first_true_index(
            cross_long
        )
    )

    first_approach_near_short = (
        first_true_index(
            approach_near_short
        )
    )

    first_approach_near_long = (
        first_true_index(
            approach_near_long
        )
    )

    last_near = (
        last_true_index(
            near
        )
    )

    first_inside_observation = (
        inside[
            :,
            0
        ]
    )

    ever_inside = (
        inside.any(
            axis=1
        )
    )

    ever_near = (
        near.any(
            axis=1
        )
    )

    # Whether any inside state occurs at or after each time.
    suffix_inside = (
        np.maximum.accumulate(
            inside[
                :,
                ::-1
            ],
            axis=1,
        )[
            :,
            ::-1
        ]
        .astype(
            bool
        )
    )

    # --------------------------------------------------------
    # Valid Held support candidates.
    # --------------------------------------------------------

    valid_hold_approach = np.zeros_like(
        approach_near_short,
        dtype=bool,
    )

    if n_times >= 3:

        transition_j = np.arange(
            n_times - 1
        )

        # transition j -> j+1 must have at least t_(j+2)
        has_followup = (
            transition_j
            <= n_times - 3
        )

        # No clearly-inside state from challenge observation onward.
        no_later_inside = (
            ~suffix_inside[
                :,
                1:
            ]
        )

        valid_hold_approach = (
            approach_near_short
            & has_followup[
                np.newaxis,
                :
            ]
            & no_later_inside
        )

    first_valid_hold = (
        first_true_index(
            valid_hold_approach
        )
    )

    evidence = np.full(
        n_segments,
        "insufficient_temporal_evidence",
        dtype=object,
    )

    reason = np.full(
        n_segments,
        "no_observed_challenge",
        dtype=object,
    )

    event_index = np.full(
        n_segments,
        -1,
        dtype=np.int32,
    )

    # ========================================================
    # BURNED OVER
    # ========================================================

    burned = (
        labels
        == "Burned Over"
    )

    burned_supported = (
        burned
        & (
            first_cross_short
            >= 0
        )
    )

    evidence[
        burned_supported
    ] = "temporally_supported"

    reason[
        burned_supported
    ] = "outside_to_inside_transition"

    event_index[
        burned_supported
    ] = first_cross_short[
        burned_supported
    ]

    burned_first_inside = (
        burned
        & ~burned_supported
        & first_inside_observation
    )

    reason[
        burned_first_inside
    ] = "first_observation_already_inside"

    burned_long_gap = (
        burned
        & ~burned_supported
        & ~burned_first_inside
        & (
            first_cross_long
            >= 0
        )
    )

    reason[
        burned_long_gap
    ] = "outside_to_inside_gap_too_large"

    event_index[
        burned_long_gap
    ] = first_cross_long[
        burned_long_gap
    ]

    burned_inside_other = (
        burned
        & ~burned_supported
        & ~burned_first_inside
        & ~burned_long_gap
        & ever_inside
    )

    reason[
        burned_inside_other
    ] = "inside_without_observed_transition"

    burned_last_challenge = (
        burned
        & ~ever_inside
        & (
            last_near
            == n_times - 1
        )
    )

    reason[
        burned_last_challenge
    ] = "challenge_at_last_observation"

    burned_near_without_inside = (
        burned
        & ~ever_inside
        & ever_near
        & ~burned_last_challenge
    )

    evidence[
        burned_near_without_inside
    ] = "trajectory_conflict"

    reason[
        burned_near_without_inside
    ] = "approached_but_never_inside"

    # ========================================================
    # HELD
    # ========================================================

    held = (
        labels
        == "Held"
    )

    held_cross = (
        held
        & (
            first_cross_short
            >= 0
        )
    )

    evidence[
        held_cross
    ] = "trajectory_conflict"

    reason[
        held_cross
    ] = "held_but_crossed_inside"

    event_index[
        held_cross
    ] = first_cross_short[
        held_cross
    ]

    held_first_inside = (
        held
        & ~held_cross
        & first_inside_observation
    )

    evidence[
        held_first_inside
    ] = "trajectory_conflict"

    reason[
        held_first_inside
    ] = "first_observation_already_inside"

    held_supported = (
        held
        & ~held_cross
        & ~held_first_inside
        & (
            first_valid_hold
            >= 0
        )
    )

    evidence[
        held_supported
    ] = "temporally_supported"

    reason[
        held_supported
    ] = "outside_to_near_without_later_engulfment"

    event_index[
        held_supported
    ] = first_valid_hold[
        held_supported
    ]

    held_inside_other = (
        held
        & ~held_cross
        & ~held_first_inside
        & ~held_supported
        & ever_inside
    )

    evidence[
        held_inside_other
    ] = "trajectory_conflict"

    reason[
        held_inside_other
    ] = "midpoint_observed_inside"

    held_last_challenge = (
        held
        & ~held_cross
        & ~held_first_inside
        & ~held_supported
        & ~ever_inside
        & (
            last_near
            == n_times - 1
        )
    )

    reason[
        held_last_challenge
    ] = "challenge_at_last_observation"

    held_long_gap = (
        held
        & ~held_cross
        & ~held_first_inside
        & ~held_supported
        & ~ever_inside
        & ~held_last_challenge
        & (
            first_approach_near_long
            >= 0
        )
    )

    reason[
        held_long_gap
    ] = "approach_gap_too_large"

    event_index[
        held_long_gap
    ] = first_approach_near_long[
        held_long_gap
    ]

    held_near_other = (
        held
        & ~held_cross
        & ~held_first_inside
        & ~held_supported
        & ~ever_inside
        & ~held_last_challenge
        & ~held_long_gap
        & ever_near
    )

    reason[
        held_near_other
    ] = "near_without_resolved_approach"

    result = pd.DataFrame(
        {
            "evidence": evidence,
            "diagnostic_reason": reason,
            "first_observation_inside": first_inside_observation,
            "ever_inside": ever_inside,
            "ever_near": ever_near,
            "min_signed_distance_m": np.min(
                signed,
                axis=1,
            ),
            "min_abs_distance_m": np.min(
                np.abs(
                    signed
                ),
                axis=1,
            ),
            "first_signed_distance_m": signed[
                :,
                0
            ],
            "last_signed_distance_m": signed[
                :,
                -1
            ],
            "nirops_observation_count": n_times,
        }
    )

    if return_event_times:

        event_start = np.full(
            n_segments,
            np.datetime64(
                "NaT",
                "ns",
            ),
            dtype="datetime64[ns]",
        )

        event_end = np.full(
            n_segments,
            np.datetime64(
                "NaT",
                "ns",
            ),
            dtype="datetime64[ns]",
        )

        event_gap_hours = np.full(
            n_segments,
            np.nan,
            dtype=np.float32,
        )

        valid_event_rows = np.flatnonzero(
            event_index
            >= 0
        )

        for row_index in (
            valid_event_rows
        ):

            j = int(
                event_index[
                    row_index
                ]
            )

            if (
                j + 1
                < n_times
            ):

                event_start[
                    row_index
                ] = (
                    times[
                        j
                    ].to_datetime64()
                )

                event_end[
                    row_index
                ] = (
                    times[
                        j + 1
                    ].to_datetime64()
                )

                event_gap_hours[
                    row_index
                ] = gaps_hours[
                    j
                ]

        result[
            "challenge_interval_start_utc"
        ] = pd.to_datetime(
            event_start,
            utc=True,
        )

        result[
            "challenge_interval_end_utc"
        ] = pd.to_datetime(
            event_end,
            utc=True,
        )

        result[
            "challenge_interval_hours"
        ] = event_gap_hours

    return result


## 13. Per-incident paths and resumability checks

In [ ]:

def incident_paths(
    irwin,
):

    return {
        "segments": (
            SEGMENT_ROOT
            / f"{irwin}.parquet"
        ),
        "trajectory": (
            TRAJECTORY_ROOT
            / f"{irwin}.npz"
        ),
        "primary": (
            PRIMARY_ROOT
            / f"{irwin}.parquet"
        ),
        "supported": (
            SUPPORTED_ROOT
            / f"{irwin}.parquet"
        ),
        "sensitivity": (
            SENSITIVITY_ROOT
            / f"{irwin}.csv"
        ),
        "metadata": (
            INCIDENT_META_ROOT
            / f"{irwin}.json"
        ),
        "observation_runtime": (
            INCIDENT_META_ROOT
            / f"{irwin}_observation_runtime.csv"
        ),
    }


def incident_complete(
    irwin,
):

    paths = (
        incident_paths(
            irwin
        )
    )

    required = [
        paths[
            "segments"
        ],
        paths[
            "trajectory"
        ],
        paths[
            "primary"
        ],
        paths[
            "supported"
        ],
        paths[
            "sensitivity"
        ],
        paths[
            "metadata"
        ],
    ]

    return all(
        path.exists()
        for path in required
    )



# 14. Sensitivity summary helper

The row-level trajectory is saved only once.

The 12 threshold combinations are evaluated from the same cached matrix:

\[
4 \text{ spatial tolerances}
\times
3 \text{ temporal gaps}
=
12.
\]

To avoid creating an unnecessarily huge 12× duplicated segment table, sensitivity results are stored as aggregated incident-level counts.

The full row-level output is retained for the primary 50 m / 48 h setting.


In [ ]:

def build_sensitivity_summary(
    segments,
    signed,
    times,
):

    rows = []

    labels = (
        segments[
            "_engagement_norm"
        ]
        .to_numpy()
    )

    for tolerance_m in (
        SPATIAL_TOLERANCES_M
    ):

        for gap_hours in (
            TEMPORAL_GAPS_HOURS
        ):

            classified = (
                classify_temporal(
                    labels=labels,
                    signed=signed,
                    times=times,
                    tolerance_m=tolerance_m,
                    max_gap_hours=gap_hours,
                    return_event_times=False,
                )
            )

            temp = pd.DataFrame(
                {
                    "_engagement_norm": labels,
                    "evidence": classified[
                        "evidence"
                    ].to_numpy(),
                    "diagnostic_reason": classified[
                        "diagnostic_reason"
                    ].to_numpy(),
                }
            )

            summary = (
                temp
                .groupby(
                    [
                        "_engagement_norm",
                        "evidence",
                        "diagnostic_reason",
                    ],
                    dropna=False,
                )
                .size()
                .rename(
                    "segments"
                )
                .reset_index()
            )

            summary[
                "tolerance_m"
            ] = tolerance_m

            summary[
                "max_gap_hours"
            ] = gap_hours

            rows.append(
                summary
            )

    return pd.concat(
        rows,
        ignore_index=True,
    )



# 15. Production run

This is the main long-running cell.

The outer progress bar advances once per incident.

A second temporary progress bar advances through NIROPS observations inside the currently running incident, so a difficult large fire does not appear frozen.

Completed incidents are written immediately and can be skipped on rerun when `RESUME=True`.


In [ ]:

RUN_LOG_PATH = (
    OUTPUT_ROOT
    / "PRODUCTION_RUN_LOG_V2.csv"
)

run_log_rows = []

outer = tqdm(
    PRODUCTION_IDS,
    total=len(
        PRODUCTION_IDS
    ),
    desc="03G_v2 production incidents",
)

for incident_number, irwin in enumerate(
    outer,
    start=1,
):

    inventory_row = (
        incident_inventory[
            incident_inventory[
                "_irwin_norm"
            ] == irwin
        ]
        .iloc[0]
    )

    incident_name = str(
        inventory_row[
            "incident_name"
        ]
    )

    # --------------------------------------------------------
    # Skip only genuinely complete incidents.
    # A v1 run that created only the segment shard is NOT
    # considered complete and will resume from that shard.
    # --------------------------------------------------------

    if (
        RESUME
        and incident_complete(
            irwin
        )
    ):

        run_log_rows.append(
            {
                "_irwin_norm": irwin,
                "incident_name": incident_name,
                "incident_number": incident_number,
                "status": "SKIPPED_COMPLETE",
                "segment_source": "existing_complete",
                "parent_lines": int(
                    inventory_row[
                        "engaged_parent_rows"
                    ]
                ),
                "segments": np.nan,
                "nirops_times": int(
                    inventory_row[
                        "nirops_times"
                    ]
                ),
                "raster_mode": np.nan,
                "regions": np.nan,
                "segmentation_seconds": 0.0,
                "trajectory_seconds": np.nan,
                "classification_seconds": np.nan,
                "sensitivity_seconds": np.nan,
                "total_seconds": 0.0,
                "error": "",
            }
        )

        outer.set_postfix(
            {
                "incident": incident_name[:16],
                "status": "resume-skip",
            }
        )

        pd.DataFrame(
            run_log_rows
        ).to_csv(
            RUN_LOG_PATH,
            index=False,
        )

        continue

    incident_start = (
        time.perf_counter()
    )

    try:

        # ----------------------------------------------------
        # 1. INCIDENT SOURCE DATA
        # ----------------------------------------------------

        incident_lines = (
            engaged_fireline[
                engaged_fireline[
                    "_irwin_norm"
                ] == irwin
            ]
            .copy()
        )

        incident_nirops = (
            matched_nirops_metric[
                matched_nirops_metric[
                    "_irwin_norm"
                ] == irwin
            ]
            .copy()
        )

        paths = (
            incident_paths(
                irwin
            )
        )

        # ----------------------------------------------------
        # 2. SEGMENTS
        # ----------------------------------------------------
        # V2 reuses the v1 segment shard if it exists.
        # All 415 failed v1 incidents wrote this shard before
        # the _time error occurred.

        segmentation_start = (
            time.perf_counter()
        )

        if (
            RESUME
            and paths[
                "segments"
            ].exists()
        ):

            segments = (
                gpd.read_parquet(
                    paths[
                        "segments"
                    ]
                )
            )

            segment_source = (
                "reused_existing_shard"
            )

        else:

            segments = (
                segment_incident(
                    incident_lines
                )
            )

            if not len(
                segments
            ):

                raise RuntimeError(
                    "No valid 250 m segment points were created."
                )

            segments.to_parquet(
                paths[
                    "segments"
                ],
                index=False,
            )

            segment_source = (
                "newly_segmented"
            )

        segmentation_seconds = (
            time.perf_counter()
            - segmentation_start
        )

        if not len(
            segments
        ):

            raise RuntimeError(
                "Segment shard exists but contains zero rows."
            )

        # Ensure the exact row order used for the trajectory matrix.
        if (
            "segment_row_index"
            in segments.columns
        ):

            segments = (
                segments
                .sort_values(
                    "segment_row_index"
                )
                .reset_index(
                    drop=True
                )
            )

        # ----------------------------------------------------
        # 3. SIGNED NIROPS TRAJECTORY
        # ----------------------------------------------------

        trajectory_start = (
            time.perf_counter()
        )

        trajectory = (
            build_incident_trajectory(
                segment_points=segments,
                incident_nirops=incident_nirops,
                incident_label=incident_name,
            )
        )

        trajectory_seconds = (
            time.perf_counter()
            - trajectory_start
        )

        if (
            trajectory[
                "status"
            ]
            != "OK"
        ):

            raise RuntimeError(
                f"Trajectory status: {trajectory['status']}"
            )

        signed = (
            trajectory[
                "signed"
            ]
        )

        times = (
            trajectory[
                "times"
            ]
        )

        # ----------------------------------------------------
        # 4. CACHE TRAJECTORY BEFORE THRESHOLDING
        # ----------------------------------------------------

        np.savez_compressed(
            paths[
                "trajectory"
            ],
            signed=signed,
            times_ns=(
                times.view(
                    "int64"
                )
            ),
        )

        trajectory[
            "observation_runtime"
        ].to_csv(
            paths[
                "observation_runtime"
            ],
            index=False,
        )

        # ----------------------------------------------------
        # 5. PRIMARY 50 m / 48 h CLASSIFICATION
        # ----------------------------------------------------

        classification_start = (
            time.perf_counter()
        )

        primary_classification = (
            classify_temporal(
                labels=segments[
                    "_engagement_norm"
                ].to_numpy(),
                signed=signed,
                times=times,
                tolerance_m=PRIMARY_TOLERANCE_M,
                max_gap_hours=PRIMARY_MAX_GAP_HOURS,
                return_event_times=True,
            )
        )

        primary = pd.concat(
            [
                segments.drop(
                    columns=[
                        "geometry",
                    ]
                ).reset_index(
                    drop=True
                ),
                primary_classification.reset_index(
                    drop=True
                ),
            ],
            axis=1,
        )

        primary[
            "primary_tolerance_m"
        ] = PRIMARY_TOLERANCE_M

        primary[
            "primary_max_gap_hours"
        ] = PRIMARY_MAX_GAP_HOURS

        primary.to_parquet(
            paths[
                "primary"
            ],
            index=False,
        )

        supported = (
            primary[
                primary[
                    "evidence"
                ] == "temporally_supported"
            ]
            .copy()
        )

        supported.to_parquet(
            paths[
                "supported"
            ],
            index=False,
        )

        classification_seconds = (
            time.perf_counter()
            - classification_start
        )

        # ----------------------------------------------------
        # 6. SENSITIVITY FROM SAME CACHED MATRIX
        # ----------------------------------------------------

        sensitivity_start = (
            time.perf_counter()
        )

        sensitivity = (
            build_sensitivity_summary(
                segments=segments,
                signed=signed,
                times=times,
            )
        )

        sensitivity[
            "_irwin_norm"
        ] = irwin

        sensitivity[
            "incident_name"
        ] = incident_name

        sensitivity.to_csv(
            paths[
                "sensitivity"
            ],
            index=False,
        )

        sensitivity_seconds = (
            time.perf_counter()
            - sensitivity_start
        )

        # ----------------------------------------------------
        # 7. INCIDENT METADATA
        # ----------------------------------------------------

        region_info = (
            trajectory[
                "region_info"
            ]
        )

        metadata = {
            "notebook_build_id": NOTEBOOK_BUILD_ID,
            "_irwin_norm": irwin,
            "incident_name": incident_name,
            "year": (
                int(
                    inventory_row[
                        "year"
                    ]
                )
                if pd.notna(
                    inventory_row[
                        "year"
                    ]
                )
                else None
            ),
            "segment_source": segment_source,
            "parent_line_rows": int(
                len(
                    incident_lines
                )
            ),
            "segments": int(
                len(
                    segments
                )
            ),
            "held_segments": int(
                (
                    segments[
                        "_engagement_norm"
                    ]
                    == "Held"
                ).sum()
            ),
            "burned_over_segments": int(
                (
                    segments[
                        "_engagement_norm"
                    ]
                    == "Burned Over"
                ).sum()
            ),
            "nirops_times": int(
                len(
                    times
                )
            ),
            "raster_mode": region_info[
                "mode"
            ],
            "single_grid_cells_if_used": int(
                region_info[
                    "single_grid_cells"
                ]
            ),
            "processing_regions": int(
                len(
                    region_info[
                        "regions"
                    ]
                )
            ),
            "supported_segments_primary": int(
                len(
                    supported
                )
            ),
            "segmentation_seconds": segmentation_seconds,
            "trajectory_seconds": trajectory_seconds,
            "classification_seconds": classification_seconds,
            "sensitivity_seconds": sensitivity_seconds,
        }

        paths[
            "metadata"
        ].write_text(
            json.dumps(
                metadata,
                indent=2,
            ),
            encoding="utf-8",
        )

        total_seconds = (
            time.perf_counter()
            - incident_start
        )

        run_log_rows.append(
            {
                "_irwin_norm": irwin,
                "incident_name": incident_name,
                "incident_number": incident_number,
                "status": "OK",
                "segment_source": segment_source,
                "parent_lines": len(
                    incident_lines
                ),
                "segments": len(
                    segments
                ),
                "nirops_times": len(
                    times
                ),
                "raster_mode": region_info[
                    "mode"
                ],
                "regions": len(
                    region_info[
                        "regions"
                    ]
                ),
                "segmentation_seconds": segmentation_seconds,
                "trajectory_seconds": trajectory_seconds,
                "classification_seconds": classification_seconds,
                "sensitivity_seconds": sensitivity_seconds,
                "total_seconds": total_seconds,
                "error": "",
            }
        )

        outer.set_postfix(
            {
                "incident": incident_name[:14],
                "seg": f"{len(segments):,}",
                "source": (
                    "reuse"
                    if segment_source
                    == "reused_existing_shard"
                    else "new"
                ),
                "mode": region_info[
                    "mode"
                ],
                "sec": f"{total_seconds:.1f}",
            }
        )

        del signed
        del trajectory
        del primary_classification
        del primary
        del supported
        del sensitivity
        del segments

        gc.collect()

    except Exception as exc:

        total_seconds = (
            time.perf_counter()
            - incident_start
        )

        error_text = (
            f"{type(exc).__name__}: {exc}"
        )

        run_log_rows.append(
            {
                "_irwin_norm": irwin,
                "incident_name": incident_name,
                "incident_number": incident_number,
                "status": "ERROR",
                "segment_source": np.nan,
                "parent_lines": int(
                    inventory_row[
                        "engaged_parent_rows"
                    ]
                ),
                "segments": np.nan,
                "nirops_times": int(
                    inventory_row[
                        "nirops_times"
                    ]
                ),
                "raster_mode": np.nan,
                "regions": np.nan,
                "segmentation_seconds": np.nan,
                "trajectory_seconds": np.nan,
                "classification_seconds": np.nan,
                "sensitivity_seconds": np.nan,
                "total_seconds": total_seconds,
                "error": error_text,
            }
        )

        error_path = (
            INCIDENT_META_ROOT
            / f"{irwin}_ERROR_V2.txt"
        )

        error_path.write_text(
            traceback.format_exc(),
            encoding="utf-8",
        )

        outer.set_postfix(
            {
                "incident": incident_name[:14],
                "status": "ERROR",
            }
        )

        gc.collect()

    pd.DataFrame(
        run_log_rows
    ).to_csv(
        RUN_LOG_PATH,
        index=False,
    )


print()
print("=" * 78)
print("03G_v2 PRODUCTION LOOP COMPLETE")
print("=" * 78)

display(
    pd.DataFrame(
        run_log_rows
    )
)



# 16. Rebuild the complete manifest from disk

This step is intentionally independent of the current Python session.

Therefore the final report works after:

- a completed fresh run;
- an interrupted run that was resumed;
- reopening the notebook on another day.


In [ ]:

metadata_files = sorted(
    INCIDENT_META_ROOT.glob(
        "*.json"
    )
)

metadata_rows = []

for path in metadata_files:

    try:

        metadata_rows.append(
            json.loads(
                path.read_text(
                    encoding="utf-8"
                )
            )
        )

    except Exception as exc:

        print(
            "Metadata read failed:",
            path.name,
            exc,
        )

production_manifest = (
    pd.DataFrame(
        metadata_rows
    )
)

if len(
    production_manifest
):

    production_manifest = (
        production_manifest
        .sort_values(
            [
                "year",
                "incident_name",
            ]
        )
        .reset_index(
            drop=True
        )
    )

display(
    production_manifest
)

production_manifest.to_csv(
    OUTPUT_ROOT
    / "PRODUCTION_MANIFEST.csv",
    index=False,
)

print(
    "Completed incident metadata files:",
    len(
        production_manifest
    )
)

print(
    "Target production incidents:",
    len(
        PRODUCTION_IDS
    )
)


## 17. Aggregate primary 50 m / 48 h evidence

In [ ]:

primary_files = sorted(
    PRIMARY_ROOT.glob(
        "*.parquet"
    )
)

primary_summary_rows = []

for path in tqdm(
    primary_files,
    desc="Aggregate primary shards",
):

    shard = pd.read_parquet(
        path,
        columns=[
            "_irwin_norm",
            "_engagement_norm",
            "evidence",
            "diagnostic_reason",
            "actual_segment_length_m",
        ],
    )

    summary = (
        shard
        .groupby(
            [
                "_engagement_norm",
                "evidence",
                "diagnostic_reason",
            ],
            dropna=False,
        )
        .agg(
            segments=(
                "evidence",
                "size",
            ),
            segment_length_km=(
                "actual_segment_length_m",
                lambda s: (
                    s.sum()
                    / 1000.0
                ),
            ),
        )
        .reset_index()
    )

    summary[
        "_irwin_norm"
    ] = (
        shard[
            "_irwin_norm"
        ].iloc[0]
    )

    primary_summary_rows.append(
        summary
    )

    del shard


primary_incident_summary = (
    pd.concat(
        primary_summary_rows,
        ignore_index=True,
    )
    if primary_summary_rows
    else pd.DataFrame()
)

if len(
    primary_incident_summary
):

    primary_global_summary = (
        primary_incident_summary
        .groupby(
            [
                "_engagement_norm",
                "evidence",
                "diagnostic_reason",
            ],
            dropna=False,
        )
        .agg(
            segments=(
                "segments",
                "sum",
            ),
            incidents=(
                "_irwin_norm",
                "nunique",
            ),
            segment_length_km=(
                "segment_length_km",
                "sum",
            ),
        )
        .reset_index()
    )

    primary_global_summary[
        "pct_within_label"
    ] = (
        primary_global_summary[
            "segments"
        ]
        / primary_global_summary.groupby(
            "_engagement_norm"
        )[
            "segments"
        ].transform(
            "sum"
        )
        * 100
    )

else:

    primary_global_summary = (
        pd.DataFrame()
    )

display(
    primary_global_summary
)

primary_incident_summary.to_csv(
    OUTPUT_ROOT
    / "primary_incident_evidence_summary.csv",
    index=False,
)

primary_global_summary.to_csv(
    OUTPUT_ROOT
    / "primary_global_evidence_summary.csv",
    index=False,
)


## 18. Aggregate 12-setting sensitivity

In [ ]:

sensitivity_files = sorted(
    SENSITIVITY_ROOT.glob(
        "*.csv"
    )
)

sensitivity_frames = []

for path in tqdm(
    sensitivity_files,
    desc="Aggregate sensitivity shards",
):

    sensitivity_frames.append(
        pd.read_csv(
            path
        )
    )

sensitivity_all = (
    pd.concat(
        sensitivity_frames,
        ignore_index=True,
    )
    if sensitivity_frames
    else pd.DataFrame()
)

if len(
    sensitivity_all
):

    sensitivity_global = (
        sensitivity_all
        .groupby(
            [
                "tolerance_m",
                "max_gap_hours",
                "_engagement_norm",
                "evidence",
                "diagnostic_reason",
            ],
            dropna=False,
        )
        .agg(
            segments=(
                "segments",
                "sum",
            ),
            incidents=(
                "_irwin_norm",
                "nunique",
            ),
        )
        .reset_index()
    )

    sensitivity_global[
        "pct_within_setting_label"
    ] = (
        sensitivity_global[
            "segments"
        ]
        / sensitivity_global.groupby(
            [
                "tolerance_m",
                "max_gap_hours",
                "_engagement_norm",
            ]
        )[
            "segments"
        ].transform(
            "sum"
        )
        * 100
    )

else:

    sensitivity_global = (
        pd.DataFrame()
    )

display(
    sensitivity_global
)

sensitivity_global.to_csv(
    OUTPUT_ROOT
    / "global_spatial_temporal_sensitivity.csv",
    index=False,
)


## 19. Compact sensitivity support/conflict table

In [ ]:

if len(
    sensitivity_global
):

    compact_rows = []

    settings = (
        sensitivity_global[
            [
                "tolerance_m",
                "max_gap_hours",
            ]
        ]
        .drop_duplicates()
        .sort_values(
            [
                "tolerance_m",
                "max_gap_hours",
            ]
        )
    )

    for setting in (
        settings.itertuples(
            index=False
        )
    ):

        for label in (
            ENGAGED_LABELS
        ):

            subset = (
                sensitivity_global[
                    (
                        sensitivity_global[
                            "tolerance_m"
                        ]
                        == setting.tolerance_m
                    )
                    & (
                        sensitivity_global[
                            "max_gap_hours"
                        ]
                        == setting.max_gap_hours
                    )
                    & (
                        sensitivity_global[
                            "_engagement_norm"
                        ]
                        == label
                    )
                ]
            )

            total = (
                subset[
                    "segments"
                ].sum()
            )

            supported = (
                subset.loc[
                    subset[
                        "evidence"
                    ]
                    == "temporally_supported",
                    "segments",
                ].sum()
            )

            conflict = (
                subset.loc[
                    subset[
                        "evidence"
                    ]
                    == "trajectory_conflict",
                    "segments",
                ].sum()
            )

            insufficient = (
                subset.loc[
                    subset[
                        "evidence"
                    ]
                    == "insufficient_temporal_evidence",
                    "segments",
                ].sum()
            )

            compact_rows.append(
                {
                    "tolerance_m": setting.tolerance_m,
                    "max_gap_hours": setting.max_gap_hours,
                    "_engagement_norm": label,
                    "segments": total,
                    "supported": supported,
                    "conflict": conflict,
                    "insufficient": insufficient,
                    "support_pct": (
                        supported
                        / total
                        * 100
                        if total
                        else np.nan
                    ),
                    "conflict_pct": (
                        conflict
                        / total
                        * 100
                        if total
                        else np.nan
                    ),
                    "insufficient_pct": (
                        insufficient
                        / total
                        * 100
                        if total
                        else np.nan
                    ),
                }
            )

    compact_sensitivity = (
        pd.DataFrame(
            compact_rows
        )
    )

else:

    compact_sensitivity = (
        pd.DataFrame()
    )

display(
    compact_sensitivity
)

compact_sensitivity.to_csv(
    OUTPUT_ROOT
    / "compact_spatial_temporal_sensitivity.csv",
    index=False,
)


## 20. Production coverage summary

In [ ]:

target_ids = set(
    PRODUCTION_IDS
)

completed_ids = set(
    production_manifest[
        "_irwin_norm"
    ].tolist()
) if len(
    production_manifest
) else set()

missing_ids = (
    target_ids
    - completed_ids
)

coverage_summary = pd.DataFrame(
    [
        {
            "target_incidents": len(
                target_ids
            ),
            "completed_incidents": len(
                completed_ids
            ),
            "missing_or_failed_incidents": len(
                missing_ids
            ),
            "completion_pct": (
                len(
                    completed_ids
                )
                / len(
                    target_ids
                )
                * 100
                if target_ids
                else np.nan
            ),
            "total_segments_completed": (
                production_manifest[
                    "segments"
                ].sum()
                if len(
                    production_manifest
                )
                else 0
            ),
            "primary_supported_segments": (
                production_manifest[
                    "supported_segments_primary"
                ].sum()
                if len(
                    production_manifest
                )
                else 0
            ),
            "single_grid_incidents": (
                (
                    production_manifest[
                        "raster_mode"
                    ]
                    == "single"
                ).sum()
                if len(
                    production_manifest
                )
                else 0
            ),
            "tiled_incidents": (
                (
                    production_manifest[
                        "raster_mode"
                    ]
                    == "tiled"
                ).sum()
                if len(
                    production_manifest
                )
                else 0
            ),
        }
    ]
)

display(
    coverage_summary
)

coverage_summary.to_csv(
    OUTPUT_ROOT
    / "production_coverage_summary.csv",
    index=False,
)

if missing_ids:

    missing_table = (
        incident_inventory[
            incident_inventory[
                "_irwin_norm"
            ].isin(
                missing_ids
            )
        ]
        .copy()
    )

    display(
        missing_table
    )

    missing_table.to_csv(
        OUTPUT_ROOT
        / "missing_or_failed_incidents.csv",
        index=False,
    )



# 21. Final dataset schema / usage notes

The production dataset is intentionally sharded by incident.

### Segment metadata

```text
data/processed/fireline_dynamic_250m/segments/<IRWIN>.parquet
```

Each row is a 250 m-or-shorter local segment unit.

### Signed trajectory cache

```text
data/processed/fireline_dynamic_250m/trajectories/<IRWIN>.npz
```

Contains:

```text
signed     [N_segments, N_nirops_times]
times_ns   [N_nirops_times]
```

Rows align with `segment_row_index` in the segment Parquet shard.

### Primary temporal classification

```text
data/processed/fireline_dynamic_250m/primary_50m_48h/<IRWIN>.parquet
```

Contains all segment rows and primary evidence classifications.

### Temporally supported subset

```text
data/processed/fireline_dynamic_250m/temporally_supported_50m_48h/<IRWIN>.parquet
```

This is the main candidate subset for subsequent weather / terrain / fuel feature extraction.

The subset should still be described as **temporally corroborated quality-assured outcomes**, not direct field-observed physical line success/failure.


In [ ]:

schema_notes = {
    "hierarchy": [
        "incident",
        "parent_fireline",
        "250m_segment",
        "nirops_temporal_trajectory",
    ],
    "primary_modeling_unit": "250 m-or-shorter segment",
    "primary_spatial_tolerance_m": PRIMARY_TOLERANCE_M,
    "primary_max_gap_hours": PRIMARY_MAX_GAP_HOURS,
    "raster_resolution_m": RASTER_RESOLUTION_M,
    "trajectory_signed_distance_convention": {
        "positive": "outside observed burned area",
        "negative": "inside observed burned area",
        "capped_absolute_distance_m": FAR_DISTANCE_M,
    },
    "recommended_validation_group": "_irwin_norm",
    "critical_leakage_rule": (
        "Never randomly split segments from the same incident "
        "between training and testing."
    ),
}

SCHEMA_NOTES_PATH = (
    PROCESSED_ROOT
    / "DATASET_NOTES.json"
)

SCHEMA_NOTES_PATH.write_text(
    json.dumps(
        schema_notes,
        indent=2,
    ),
    encoding="utf-8",
)

print(
    json.dumps(
        schema_notes,
        indent=2,
    )
)


## 22. Write production report

In [ ]:

def df_to_md(
    dataframe,
    max_rows=200,
):

    if (
        dataframe is None
        or len(
            dataframe
        ) == 0
    ):
        return "_No rows._"

    try:

        return (
            dataframe
            .head(
                max_rows
            )
            .to_markdown(
                index=False
            )
        )

    except Exception:

        return (
            dataframe
            .head(
                max_rows
            )
            .to_string(
                index=False
            )
        )


report = f"""# Fireline Temporal Production Reconstruction — 03G_v2

Notebook build: `{NOTEBOOK_BUILD_ID}`

Generated: {datetime.now(timezone.utc).isoformat()}

## Fixed production settings

Primary segment length: **{SEGMENT_LENGTH_M} m**

Raster resolution: **{RASTER_RESOLUTION_M} m**

Primary challenge tolerance: **{PRIMARY_TOLERANCE_M} m**

Primary maximum NIROPS interval: **{PRIMARY_MAX_GAP_HOURS} h**

Sensitivity tolerances:

{SPATIAL_TOLERANCES_M}

Sensitivity maximum gaps:

{TEMPORAL_GAPS_HOURS}

## Coverage

{df_to_md(coverage_summary)}

## Primary 50 m / 48 h evidence

{df_to_md(primary_global_summary)}

## Compact sensitivity

{df_to_md(compact_sensitivity)}

## Raster architecture

Incidents use one local 25 m raster whenever the incident grid is at most {MAX_SINGLE_GRID_CELLS:,} cells.

Larger incidents automatically switch to {TILE_CORE_SIZE_M / 1000:.1f} km core tiles with a {TILE_HALO_M:.0f} m halo.

This preserves the 25 m production raster resolution rather than coarsening geographically large incidents.

## Scientific interpretation

The original Fireline Held/Burned Over outcomes remain quality-assured spatially inferred outcomes.

NIROPS progression temporally corroborates a subset of those outcomes.

Challenge time is interval-censored between consecutive NIROPS observations and should not be interpreted as an exact physical contact or failure time.

## Output hierarchy

- incident
- parent operational fireline
- 250 m local segment
- signed NIROPS distance trajectory

## Recommended next stage

Do not train the final ML model until production coverage and sensitivity stability have been reviewed.

If the temporal dataset is stable, the next feature stage should add:

1. HRRR weather aggregated over the inferred challenge interval;
2. local line orientation relative to observed/modelled wind;
3. terrain from 3DEP;
4. pre-event fuels/vegetation from LANDFIRE;
5. observed NIROPS approach-rate and trajectory features.

All model validation must hold out complete incidents using `_irwin_norm`.
"""

REPORT_PATH = (
    OUTPUT_ROOT
    / "TEMPORAL_PRODUCTION_REPORT.md"
)

REPORT_PATH.write_text(
    report,
    encoding="utf-8",
)

print(
    report
)

print(
    "\nSaved:",
    REPORT_PATH
)


## 23. Final output inventory

In [ ]:

print("=" * 78)
print("03G_v2 PROCESSED DATA")
print("=" * 78)

for directory in [
    SEGMENT_ROOT,
    TRAJECTORY_ROOT,
    PRIMARY_ROOT,
    SUPPORTED_ROOT,
    SENSITIVITY_ROOT,
    INCIDENT_META_ROOT,
]:

    files = [
        path
        for path in directory.iterdir()
        if path.is_file()
    ]

    total_bytes = sum(
        path.stat().st_size
        for path in files
    )

    print(
        f"{directory.name:<40}"
        f"{len(files):>8,} files"
        f"{total_bytes / (1024**2):>12,.1f} MB"
    )

print()
print("=" * 78)
print("03G_v2 REPORT OUTPUTS")
print("=" * 78)

for path in sorted(
    OUTPUT_ROOT.iterdir()
):

    if path.is_file():

        size = (
            path.stat().st_size
            / 1024
        )

        print(
            f"{path.name:<60}"
            f"{size:>12,.1f} KB"
        )

print()
print("DONE:", NOTEBOOK_BUILD_ID)
